# Train A Vanilla Transformer For CAMELS QObs Prediction

This notebook trains one shared vanilla transformer across all selected CAMELS basins to predict streamflow/runoff targets from meteorological forcing features.

Protocol used here:

- one shared transformer for all selected basins
- independent holdout period: `2008-01-01` to `2014-12-31`
- expanding-window cross-validation on years before 2008 for hyperparameter tuning
- all available preprocessed features are used
- NSE-based loss is used for training
- random search is used for hyperparameter tuning instead of grid search

Important note: the current processed dataset in this repository was generated with `target_unit = "mm/day"`, so the target column in the joined file is `QObs(mm/day)`. If you want raw `QObs(cfs)` instead, regenerate the preprocessing outputs from `prepare_camels_transformer_data.ipynb` with `TARGET_UNIT = 'cfs'`.

## Dependencies

The notebook was tested with the local `cs7643` environment. Core packages:

```bash
pip install numpy pandas pyarrow torch matplotlib tqdm
```

In [2]:
from __future__ import annotations

import copy
import json
import math
import random
import time
from bisect import bisect_right
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset

## Configuration

Edit this cell before running the notebook.

Main controls:

- `BASIN_IDS`: leave empty to use all basins in the processed file
- `LOOKBACK_DAYS`: input sequence length
- `PREDICTION_HORIZON_DAYS`: number of days ahead to predict
- `INDEPENDENT_VAL_START` and `INDEPENDENT_VAL_END`: fixed holdout period
- `CV_NUM_FOLDS`: number of expanding-window folds on years before the holdout period
- `SEARCH_TRIALS`: random-search budget for hyperparameter tuning

In [3]:
CONFIG = {
    'DATA_DIR': '/Users/xshan/Research/GT/cs7643_DL/courseProject/CAMELS_data_load/processed',
    'JOINED_FILENAME_PREFERENCE': ['camels_transformer_joined.parquet', 'camels_transformer_joined.csv'],
    'METADATA_FILENAME': 'camels_transformer_metadata.json',
    'OUTPUT_DIR': '/Users/xshan/Research/GT/cs7643_DL/courseProject/CAMELS_data_load/model_artifacts',
    'BASIN_IDS': [],
    'LOOKBACK_DAYS': 365,
    'PREDICTION_HORIZON_DAYS': 1,
    'USE_ALL_FEATURES': True,
    'INDEPENDENT_VAL_START': '2008-01-01',
    'INDEPENDENT_VAL_END': '2014-12-31',
    'CV_NUM_FOLDS': 4,
    'SEED': 7643,
    'DEVICE': 'cuda' if torch.cuda.is_available() else 'cpu',
    'NUM_WORKERS': 0,
    'PIN_MEMORY': False,
    'SEARCH_TRIALS': 8,
    'SEARCH_EPOCHS': 6,
    'SEARCH_PATIENCE': 2,
    'FINAL_MAX_EPOCHS': 20,
    'TUNING_MAX_TRAIN_WINDOWS': 50000,
    'TUNING_MAX_VAL_WINDOWS': 15000,
    'SAVE_CHECKPOINTS': True,
}

Path(CONFIG['OUTPUT_DIR']).mkdir(parents=True, exist_ok=True)
CONFIG

{'DATA_DIR': '/Users/xshan/Research/GT/cs7643_DL/courseProject/CAMELS_data_load/processed',
 'JOINED_FILENAME_PREFERENCE': ['camels_transformer_joined.parquet',
  'camels_transformer_joined.csv'],
 'METADATA_FILENAME': 'camels_transformer_metadata.json',
 'OUTPUT_DIR': '/Users/xshan/Research/GT/cs7643_DL/courseProject/CAMELS_data_load/model_artifacts',
 'BASIN_IDS': [],
 'LOOKBACK_DAYS': 365,
 'PREDICTION_HORIZON_DAYS': 1,
 'USE_ALL_FEATURES': True,
 'INDEPENDENT_VAL_START': '2008-01-01',
 'INDEPENDENT_VAL_END': '2014-12-31',
 'CV_NUM_FOLDS': 4,
 'SEED': 7643,
 'DEVICE': 'cpu',
 'NUM_WORKERS': 0,
 'PIN_MEMORY': False,
 'SEARCH_TRIALS': 8,
 'SEARCH_EPOCHS': 6,
 'SEARCH_PATIENCE': 2,
 'FINAL_MAX_EPOCHS': 20,
 'TUNING_MAX_TRAIN_WINDOWS': 50000,
 'TUNING_MAX_VAL_WINDOWS': 15000,
 'SAVE_CHECKPOINTS': True}

## Hyperparameter Search Space

This notebook uses random search over a bounded search space rather than exhaustive grid search.

In [4]:
SEARCH_SPACE = {
    'd_model': [64, 96, 128, 160],
    'nhead': [4, 8],
    'num_layers': [2, 3, 4],
    'dim_feedforward': [128, 256, 384, 512],
    'dropout': [0.05, 0.10, 0.20],
    'learning_rate': [1e-4, 2e-4, 3e-4, 5e-4],
    'weight_decay': [0.0, 1e-5, 1e-4, 5e-4],
    'batch_size': [32, 64, 96, 128],
}
SEARCH_SPACE

{'d_model': [64, 96, 128, 160],
 'nhead': [4, 8],
 'num_layers': [2, 3, 4],
 'dim_feedforward': [128, 256, 384, 512],
 'dropout': [0.05, 0.1, 0.2],
 'learning_rate': [0.0001, 0.0002, 0.0003, 0.0005],
 'weight_decay': [0.0, 1e-05, 0.0001, 0.0005],
 'batch_size': [32, 64, 96, 128]}

## Helpers

In [5]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_metadata(config: Dict) -> Dict:
    metadata_path = Path(config['DATA_DIR']) / config['METADATA_FILENAME']
    return json.loads(metadata_path.read_text(encoding='utf-8'))


def choose_joined_path(config: Dict) -> Path:
    data_dir = Path(config['DATA_DIR'])
    for filename in config['JOINED_FILENAME_PREFERENCE']:
        path = data_dir / filename
        if path.exists():
            return path
    raise FileNotFoundError('Could not find a joined CAMELS file in processed/.')


def read_joined_table(config: Dict, metadata: Dict) -> pd.DataFrame:
    joined_path = choose_joined_path(config)
    feature_columns = metadata['feature_columns']
    target_column = metadata['target_column_in_joined_file']
    columns = ['basin_id', 'date'] + feature_columns + [target_column]

    if joined_path.suffix == '.parquet':
        df = pd.read_parquet(joined_path, columns=columns)
    else:
        df = pd.read_csv(joined_path, usecols=columns, parse_dates=['date'])

    df['basin_id'] = df['basin_id'].astype(str).str.zfill(8)
    df['date'] = pd.to_datetime(df['date'])
    return df.sort_values(['basin_id', 'date']).reset_index(drop=True)


def maybe_filter_basins(df: pd.DataFrame, basin_ids: Sequence[str]) -> pd.DataFrame:
    if not basin_ids:
        return df
    basin_ids = [str(x).zfill(8) for x in basin_ids]
    return df[df['basin_id'].isin(basin_ids)].copy()


def infer_feature_columns(metadata: Dict, config: Dict) -> List[str]:
    if not config['USE_ALL_FEATURES']:
        raise ValueError('This notebook is configured to use all features. Set USE_ALL_FEATURES=True.')
    return list(metadata['feature_columns'])


def global_nse(pred: np.ndarray, target: np.ndarray, eps: float = 1e-6) -> float:
    numerator = np.sum((pred - target) ** 2)
    denominator = np.sum((target - target.mean()) ** 2) + eps
    return float(1.0 - numerator / denominator)


def nse_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    pred = pred.reshape(-1)
    target = target.reshape(-1)
    numerator = torch.sum((pred - target) ** 2)
    denominator = torch.sum((target - torch.mean(target)) ** 2) + eps
    return numerator / denominator


def regression_metrics(pred: np.ndarray, target: np.ndarray) -> Dict[str, float]:
    pred = pred.reshape(-1)
    target = target.reshape(-1)
    rmse = float(np.sqrt(np.mean((pred - target) ** 2)))
    mae = float(np.mean(np.abs(pred - target)))
    nse = global_nse(pred, target)
    return {'rmse': rmse, 'mae': mae, 'nse': nse, 'loss': 1.0 - nse}


def sample_config(search_space: Dict[str, List], seen: set, rng: random.Random) -> Dict:
    keys = sorted(search_space)
    while True:
        candidate = {key: rng.choice(search_space[key]) for key in keys}
        if candidate['d_model'] % candidate['nhead'] != 0:
            continue
        signature = tuple((key, candidate[key]) for key in keys)
        if signature not in seen:
            seen.add(signature)
            return candidate

## Load Processed CAMELS Data

In [6]:
set_seed(CONFIG['SEED'])
metadata = load_metadata(CONFIG)
feature_columns = infer_feature_columns(metadata, CONFIG)
target_column = metadata['target_column_in_joined_file']

joined_df = read_joined_table(CONFIG, metadata)
joined_df = maybe_filter_basins(joined_df, CONFIG['BASIN_IDS'])

print('Using joined file:', choose_joined_path(CONFIG))
print('Target column:', target_column)
print('Feature columns:', feature_columns)
print('Rows:', len(joined_df))
print('Basins:', joined_df['basin_id'].nunique())
print('Date range:', joined_df['date'].min().date(), 'to', joined_df['date'].max().date())

joined_df.head()

Using joined file: /Users/xshan/Research/GT/cs7643_DL/courseProject/CAMELS_data_load/processed/camels_transformer_joined.parquet
Target column: QObs(mm/day)
Feature columns: ['Dayl(s)', 'PRCP(mm/day)', 'SRAD(W/m2)', 'SWE(mm)', 'Tmax(C)', 'Tmin(C)', 'Vp(Pa)', 'doy_sin', 'doy_cos', 'basin_code']
Rows: 8415489
Basins: 671
Date range: 1980-01-01 to 2014-12-31


,basin_id,date,Dayl(s),PRCP(mm/day),SRAD(W/m2),SWE(mm),Tmax(C),Tmin(C),Vp(Pa),doy_sin,doy_cos,basin_code,QObs(mm/day)
0,01013500,1980-01-01,30172.48,0.0,218.66,0.0,-13.04,-13.04,203.28,0.017166,0.999853,0,0.711372
1,01013500,1980-01-02,30253.07,0.0,199.05,0.0,-10.93,-10.93,237.37,0.034328,0.999411,0,0.695081
2,01013500,1980-01-03,30344.16,0.0,197.64,0.0,-13.60,-13.60,169.39,0.051479,0.998674,0,0.678790
3,01013500,1980-01-04,30408.34,0.0,214.61,0.0,-16.53,-16.53,134.57,0.068615,0.997643,0,0.673359
4,01013500,1980-01-05,30413.49,0.0,206.02,0.0,-17.60,-17.60,129.30,0.085731,0.996318,0,0.657068


## Build Per-Basin Arrays And Time Splits

This notebook builds sequence windows lazily and uses:

- a fixed independent holdout period from 2008 through 2014
- expanding-window cross-validation on earlier years for hyperparameter tuning

In [7]:
def build_basin_store(df: pd.DataFrame, feature_columns: List[str], target_column: str) -> Dict[str, Dict[str, np.ndarray]]:
    store = {}
    for basin_id, basin_df in df.groupby('basin_id', sort=True):
        basin_df = basin_df.sort_values('date').reset_index(drop=True)
        store[basin_id] = {
            'features': basin_df[feature_columns].to_numpy(dtype=np.float32),
            'target': basin_df[target_column].to_numpy(dtype=np.float32),
            'dates': basin_df['date'].to_numpy(),
        }
    return store


def eligible_target_indices(n_rows: int, lookback_days: int, horizon_days: int) -> np.ndarray:
    first_target_idx = lookback_days + horizon_days - 1
    if n_rows <= first_target_idx:
        return np.array([], dtype=np.int64)
    return np.arange(first_target_idx, n_rows, dtype=np.int64)


def year_chunks_for_expanding_cv(train_years: List[int], cv_num_folds: int) -> List[List[int]]:
    chunks = np.array_split(np.array(train_years), cv_num_folds + 1)
    return [chunk.astype(int).tolist() for chunk in chunks if len(chunk) > 0]


def build_time_protocol(
    basin_store: Dict[str, Dict[str, np.ndarray]],
    lookback_days: int,
    horizon_days: int,
    independent_val_start: str,
    independent_val_end: str,
    cv_num_folds: int,
) -> Dict:
    holdout_start = np.datetime64(independent_val_start)
    holdout_end = np.datetime64(independent_val_end)

    pre_holdout_years = sorted({
        int(pd.Timestamp(date_value).year)
        for basin_data in basin_store.values()
        for date_value in basin_data['dates']
        if date_value < holdout_start
    })
    chunks = year_chunks_for_expanding_cv(pre_holdout_years, cv_num_folds)
    if len(chunks) < 2:
        raise ValueError('Not enough pre-holdout years to build expanding-window CV folds.')

    cv_folds = []
    for fold_idx in range(len(chunks) - 1):
        train_years = sorted([year for chunk in chunks[: fold_idx + 1] for year in chunk])
        val_years = sorted(chunks[fold_idx + 1])
        if not train_years or not val_years:
            continue
        cv_folds.append({
            'fold_id': fold_idx,
            'train_years': train_years,
            'val_years': val_years,
        })

    basin_protocol = {}
    usable_basins = []
    for basin_id, basin_data in basin_store.items():
        dates = basin_data['dates']
        target_indices = eligible_target_indices(len(dates), lookback_days, horizon_days)
        if len(target_indices) == 0:
            continue
        target_dates = dates[target_indices]

        pre_holdout_mask = target_dates < holdout_start
        holdout_mask = (target_dates >= holdout_start) & (target_dates <= holdout_end)
        pre_holdout_targets = target_indices[pre_holdout_mask]
        holdout_targets = target_indices[holdout_mask]

        if len(holdout_targets) == 0:
            continue

        fold_targets = []
        valid_basin = True
        for fold in cv_folds:
            train_year_set = set(fold['train_years'])
            val_year_set = set(fold['val_years'])
            train_targets = np.array([
                idx for idx in pre_holdout_targets if int(pd.Timestamp(dates[idx]).year) in train_year_set
            ], dtype=np.int64)
            val_targets = np.array([
                idx for idx in pre_holdout_targets if int(pd.Timestamp(dates[idx]).year) in val_year_set
            ], dtype=np.int64)
            if len(train_targets) == 0 or len(val_targets) == 0:
                valid_basin = False
                break
            fold_targets.append({
                'fold_id': fold['fold_id'],
                'train_targets': train_targets,
                'val_targets': val_targets,
            })

        if not valid_basin:
            continue

        basin_protocol[basin_id] = {
            'cv_folds': fold_targets,
            'holdout_targets': holdout_targets,
            'pre_holdout_targets': pre_holdout_targets,
        }
        usable_basins.append(basin_id)

    filtered_cv_folds = []
    for fold in cv_folds:
        filtered_cv_folds.append({
            'fold_id': fold['fold_id'],
            'train_years': fold['train_years'],
            'val_years': fold['val_years'],
        })

    return {
        'holdout_start': str(holdout_start),
        'holdout_end': str(holdout_end),
        'cv_folds': filtered_cv_folds,
        'basin_protocol': basin_protocol,
        'usable_basins': usable_basins,
    }


class WindowedTargetDataset(Dataset):
    def __init__(
        self,
        basin_store: Dict[str, Dict[str, np.ndarray]],
        target_map: Dict[str, np.ndarray],
        lookback_days: int,
        horizon_days: int,
        feature_mean: np.ndarray,
        feature_std: np.ndarray,
    ):
        self.basin_store = basin_store
        self.lookback_days = lookback_days
        self.horizon_days = horizon_days
        self.feature_mean = feature_mean.astype(np.float32)
        self.feature_std = feature_std.astype(np.float32)
        self.entries = []
        self.cum_counts = []
        running = 0

        for basin_id in sorted(target_map):
            target_indices = np.asarray(target_map[basin_id], dtype=np.int64)
            if len(target_indices) == 0:
                continue
            self.entries.append((basin_id, target_indices))
            running += len(target_indices)
            self.cum_counts.append(running)

        self.length = running

    def __len__(self) -> int:
        return self.length

    def __getitem__(self, idx: int):
        basin_pos = bisect_right(self.cum_counts, idx)
        basin_id, target_indices = self.entries[basin_pos]
        prev_count = 0 if basin_pos == 0 else self.cum_counts[basin_pos - 1]
        target_idx = int(target_indices[idx - prev_count])
        series = self.basin_store[basin_id]
        sequence_start = target_idx - self.horizon_days - self.lookback_days + 1
        sequence_end = sequence_start + self.lookback_days

        x = series['features'][sequence_start:sequence_end]
        x = (x - self.feature_mean) / self.feature_std
        y = float(series['target'][target_idx])
        date_value = series['dates'][target_idx]

        return {
            'x': torch.from_numpy(x).float(),
            'y': torch.tensor(y, dtype=torch.float32),
            'basin_id': basin_id,
            'target_date': str(np.datetime_as_string(date_value, unit='D')),
        }


def subset_dataset(dataset: Dataset, max_size: Optional[int], seed: int) -> Dataset:
    if max_size is None or len(dataset) <= max_size:
        return dataset
    rng = np.random.default_rng(seed)
    indices = np.sort(rng.choice(len(dataset), size=max_size, replace=False))
    return Subset(dataset, indices.tolist())


def make_target_map_for_fold(protocol: Dict, fold_id: int, split_name: str) -> Dict[str, np.ndarray]:
    target_map = {}
    for basin_id, basin_info in protocol['basin_protocol'].items():
        fold_info = next(item for item in basin_info['cv_folds'] if item['fold_id'] == fold_id)
        if split_name == 'train':
            target_map[basin_id] = fold_info['train_targets']
        elif split_name == 'val':
            target_map[basin_id] = fold_info['val_targets']
        elif split_name == 'pre_holdout_all':
            target_map[basin_id] = basin_info['pre_holdout_targets']
        elif split_name == 'holdout':
            target_map[basin_id] = basin_info['holdout_targets']
        else:
            raise ValueError('Unknown split name: ' + split_name)
    return target_map


def feature_stats_from_target_map(
    basin_store: Dict[str, Dict[str, np.ndarray]],
    target_map: Dict[str, np.ndarray],
    lookback_days: int,
    horizon_days: int,
) -> Tuple[np.ndarray, np.ndarray]:
    blocks = []
    for basin_id, targets in target_map.items():
        if len(targets) == 0:
            continue
        series = basin_store[basin_id]['features']
        max_target = int(np.max(targets))
        feature_end = max_target - horizon_days + 2
        feature_end = max(feature_end, 1)
        blocks.append(series[:feature_end])
    matrix = np.concatenate(blocks, axis=0)
    mean = matrix.mean(axis=0).astype(np.float32)
    std = matrix.std(axis=0).astype(np.float32)
    std[std < 1e-6] = 1.0
    return mean, std


basin_store = build_basin_store(joined_df, feature_columns, target_column)
protocol = build_time_protocol(
    basin_store=basin_store,
    lookback_days=CONFIG['LOOKBACK_DAYS'],
    horizon_days=CONFIG['PREDICTION_HORIZON_DAYS'],
    independent_val_start=CONFIG['INDEPENDENT_VAL_START'],
    independent_val_end=CONFIG['INDEPENDENT_VAL_END'],
    cv_num_folds=CONFIG['CV_NUM_FOLDS'],
)

print('Usable basins:', len(protocol['usable_basins']))
print('Independent holdout:', protocol['holdout_start'], 'to', protocol['holdout_end'])
print('Cross-validation folds:')
for fold in protocol['cv_folds']:
    print(' Fold', fold['fold_id'], 'train years', fold['train_years'][0], 'to', fold['train_years'][-1], '| val years', fold['val_years'][0], 'to', fold['val_years'][-1])

holdout_target_map = make_target_map_for_fold(protocol, fold_id=0, split_name='holdout')
pre_holdout_target_map = make_target_map_for_fold(protocol, fold_id=0, split_name='pre_holdout_all')
feature_mean_full, feature_std_full = feature_stats_from_target_map(
    basin_store,
    pre_holdout_target_map,
    CONFIG['LOOKBACK_DAYS'],
    CONFIG['PREDICTION_HORIZON_DAYS'],
)
full_holdout_dataset = WindowedTargetDataset(
    basin_store=basin_store,
    target_map=holdout_target_map,
    lookback_days=CONFIG['LOOKBACK_DAYS'],
    horizon_days=CONFIG['PREDICTION_HORIZON_DAYS'],
    feature_mean=feature_mean_full,
    feature_std=feature_std_full,
)

print('Holdout windows:', len(full_holdout_dataset))

Usable basins: 633
Independent holdout: 2008-01-01 to 2014-12-31
Cross-validation folds:
 Fold 0 train years 1980 to 1985 | val years 1986 to 1991
 Fold 1 train years 1980 to 1991 | val years 1992 to 1997
 Fold 2 train years 1980 to 1997 | val years 1998 to 2002
 Fold 3 train years 1980 to 2002 | val years 2003 to 2007
Holdout windows: 1598130


## Inspect One Fold And One Sample

In [8]:
example_fold_id = protocol['cv_folds'][0]['fold_id']
example_train_map = make_target_map_for_fold(protocol, example_fold_id, 'train')
example_val_map = make_target_map_for_fold(protocol, example_fold_id, 'val')
example_mean, example_std = feature_stats_from_target_map(
    basin_store,
    example_train_map,
    CONFIG['LOOKBACK_DAYS'],
    CONFIG['PREDICTION_HORIZON_DAYS'],
)
example_train_dataset = WindowedTargetDataset(
    basin_store=basin_store,
    target_map=example_train_map,
    lookback_days=CONFIG['LOOKBACK_DAYS'],
    horizon_days=CONFIG['PREDICTION_HORIZON_DAYS'],
    feature_mean=example_mean,
    feature_std=example_std,
)
example_sample = example_train_dataset[0]

print('Example fold:', example_fold_id)
print('Train windows in fold:', len(example_train_dataset))
print('Validation windows in fold:', sum(len(v) for v in example_val_map.values()))
print('x shape:', tuple(example_sample['x'].shape))
print('y:', float(example_sample['y']))
print('basin:', example_sample['basin_id'])
print('target date:', example_sample['target_date'])

Example fold: 0
Train windows in fold: 1115628
Validation windows in fold: 1386903
x shape: (365, 10)
y: 0.6353472471237183
basin: 01013500
target date: 1980-12-31


## Model Definition

In [9]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class VanillaTransformerRegressor(nn.Module):
    def __init__(self, num_features: int, d_model: int, nhead: int, num_layers: int, dim_feedforward: int, dropout: float):
        super().__init__()
        self.input_projection = nn.Linear(num_features, d_model)
        self.positional_encoding = PositionalEncoding(d_model=d_model, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation='gelu',
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, max(1, d_model // 2)),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(max(1, d_model // 2), 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_projection(x)
        x = self.positional_encoding(x)
        x = self.encoder(x)
        x = x[:, -1, :]
        return self.head(x).squeeze(-1)


def make_dataloader(dataset: Dataset, batch_size: int, shuffle: bool) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=CONFIG['NUM_WORKERS'],
        pin_memory=CONFIG['PIN_MEMORY'],
        drop_last=False,
    )

## Training Utilities

In [10]:
def run_epoch(model: nn.Module, loader: DataLoader, optimizer: Optional[torch.optim.Optimizer], device: str) -> Dict[str, float]:
    is_train = optimizer is not None
    model.train(is_train)
    preds, targets = [], []

    for batch in loader:
        x = batch['x'].to(device)
        y = batch['y'].to(device)
        if is_train:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(is_train):
            pred = model(x)
            loss = nse_loss(pred, y)
            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
        preds.append(pred.detach().cpu().numpy())
        targets.append(y.detach().cpu().numpy())

    pred_np = np.concatenate(preds)
    target_np = np.concatenate(targets)
    return regression_metrics(pred_np, target_np)


def fit_model_with_early_stopping(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    learning_rate: float,
    weight_decay: float,
    max_epochs: int,
    patience: int,
    device: str,
):
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    history = []
    best_state = None
    best_val_nse = -np.inf
    best_epoch = 0
    patience_left = patience

    for epoch in range(1, max_epochs + 1):
        train_metrics = run_epoch(model, train_loader, optimizer, device)
        val_metrics = run_epoch(model, val_loader, None, device)
        row = {
            'epoch': epoch,
            'train_loss': train_metrics['loss'],
            'train_nse': train_metrics['nse'],
            'val_loss': val_metrics['loss'],
            'val_nse': val_metrics['nse'],
            'val_rmse': val_metrics['rmse'],
            'val_mae': val_metrics['mae'],
        }
        history.append(row)
        print(row)

        if val_metrics['nse'] > best_val_nse:
            best_val_nse = val_metrics['nse']
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            patience_left = patience
        else:
            patience_left -= 1
            if patience_left <= 0:
                print('Early stopping triggered.')
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_epoch


def fit_model_fixed_epochs(
    model: nn.Module,
    train_loader: DataLoader,
    learning_rate: float,
    weight_decay: float,
    epochs: int,
    device: str,
):
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    history = []
    for epoch in range(1, epochs + 1):
        train_metrics = run_epoch(model, train_loader, optimizer, device)
        row = {
            'epoch': epoch,
            'train_loss': train_metrics['loss'],
            'train_nse': train_metrics['nse'],
        }
        history.append(row)
        print(row)
    return model, pd.DataFrame(history)

## Random Search With Expanding-Window Cross-Validation

Each random-search trial is evaluated by averaging validation NSE across the expanding-window folds on years before 2008.

In [ ]:
def evaluate_trial_on_cv(trial_config: Dict, protocol: Dict, basin_store: Dict[str, Dict[str, np.ndarray]], seed: int):
    device = CONFIG['DEVICE']
    fold_results = []

    for fold in protocol['cv_folds']:
        fold_id = fold['fold_id']
        train_map = make_target_map_for_fold(protocol, fold_id, 'train')
        val_map = make_target_map_for_fold(protocol, fold_id, 'val')
        feature_mean, feature_std = feature_stats_from_target_map(
            basin_store,
            train_map,
            CONFIG['LOOKBACK_DAYS'],
            CONFIG['PREDICTION_HORIZON_DAYS'],
        )
        train_dataset = WindowedTargetDataset(
            basin_store=basin_store,
            target_map=train_map,
            lookback_days=CONFIG['LOOKBACK_DAYS'],
            horizon_days=CONFIG['PREDICTION_HORIZON_DAYS'],
            feature_mean=feature_mean,
            feature_std=feature_std,
        )
        val_dataset = WindowedTargetDataset(
            basin_store=basin_store,
            target_map=val_map,
            lookback_days=CONFIG['LOOKBACK_DAYS'],
            horizon_days=CONFIG['PREDICTION_HORIZON_DAYS'],
            feature_mean=feature_mean,
            feature_std=feature_std,
        )
        train_dataset = subset_dataset(train_dataset, CONFIG['TUNING_MAX_TRAIN_WINDOWS'], seed + fold_id)
        val_dataset = subset_dataset(val_dataset, CONFIG['TUNING_MAX_VAL_WINDOWS'], seed + 100 + fold_id)

        train_loader = make_dataloader(train_dataset, batch_size=trial_config['batch_size'], shuffle=True)
        val_loader = make_dataloader(val_dataset, batch_size=trial_config['batch_size'], shuffle=False)

        model = VanillaTransformerRegressor(
            num_features=len(feature_columns),
            d_model=trial_config['d_model'],
            nhead=trial_config['nhead'],
            num_layers=trial_config['num_layers'],
            dim_feedforward=trial_config['dim_feedforward'],
            dropout=trial_config['dropout'],
        ).to(device)

        model, history_df, best_epoch = fit_model_with_early_stopping(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            learning_rate=trial_config['learning_rate'],
            weight_decay=trial_config['weight_decay'],
            max_epochs=CONFIG['SEARCH_EPOCHS'],
            patience=CONFIG['SEARCH_PATIENCE'],
            device=device,
        )
        best_row = history_df.loc[history_df['val_nse'].idxmax()].to_dict()
        fold_results.append({
            'fold_id': fold_id,
            'best_epoch': int(best_epoch),
            'best_val_nse': float(best_row['val_nse']),
            'best_val_rmse': float(best_row['val_rmse']),
            'best_val_mae': float(best_row['val_mae']),
        })

    fold_df = pd.DataFrame(fold_results)
    summary = {
        'cv_mean_val_nse': float(fold_df['best_val_nse'].mean()),
        'cv_std_val_nse': float(fold_df['best_val_nse'].std(ddof=0)),
        'cv_mean_val_rmse': float(fold_df['best_val_rmse'].mean()),
        'cv_mean_val_mae': float(fold_df['best_val_mae'].mean()),
        'cv_mean_best_epoch': float(fold_df['best_epoch'].mean()),
    }
    return summary, fold_df


search_results = []
fold_result_frames = []
seen_configs = set()
search_rng = random.Random(CONFIG['SEED'])
print('Training device:', CONFIG['DEVICE'])

for trial_idx in range(1, CONFIG['SEARCH_TRIALS'] + 1):
    trial_config = sample_config(SEARCH_SPACE, seen_configs, search_rng)
    print('\n=== Random search trial {}/{} ==='.format(trial_idx, CONFIG['SEARCH_TRIALS']))
    print(trial_config)
    start_time = time.time()
    summary, fold_df = evaluate_trial_on_cv(trial_config, protocol, basin_store, CONFIG['SEED'] + trial_idx)
    elapsed = time.time() - start_time

    result = dict(trial_config)
    result.update({
        'trial': trial_idx,
        'elapsed_sec': elapsed,
        'cv_mean_best_epoch': summary['cv_mean_best_epoch'],
        'cv_mean_val_nse': summary['cv_mean_val_nse'],
        'cv_std_val_nse': summary['cv_std_val_nse'],
        'cv_mean_val_rmse': summary['cv_mean_val_rmse'],
        'cv_mean_val_mae': summary['cv_mean_val_mae'],
    })
    search_results.append(result)
    fold_df = fold_df.copy()
    fold_df['trial'] = trial_idx
    fold_result_frames.append(fold_df)

search_results_df = pd.DataFrame(search_results).sort_values('cv_mean_val_nse', ascending=False).reset_index(drop=True)
cv_fold_results_df = pd.concat(fold_result_frames, ignore_index=True)
search_results_df

Training device: cpu

=== Random search trial 1/8 ===
{'batch_size': 96, 'd_model': 160, 'dim_feedforward': 256, 'dropout': 0.05, 'learning_rate': 0.0002, 'nhead': 8, 'num_layers': 4, 'weight_decay': 0.0005}


/Users/xshan/miniforge3/envs/cs7643/lib/python3.8/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


{'epoch': 1, 'train_loss': 0.9220168905063997, 'train_nse': 0.07798310949360032, 'val_loss': 0.6761185795179034, 'val_nse': 0.32388142048209656, 'val_rmse': 3.583561658859253, 'val_mae': 0.8368833065032959}
{'epoch': 2, 'train_loss': 0.8930550566398913, 'train_nse': 0.10694494336010874, 'val_loss': 0.6577559636345336, 'val_nse': 0.3422440363654664, 'val_rmse': 3.5345640182495117, 'val_mae': 0.9330353140830994}
{'epoch': 3, 'train_loss': 0.8828323532491898, 'train_nse': 0.11716764675081015, 'val_loss': 0.6323207837712479, 'val_nse': 0.3676792162287521, 'val_rmse': 3.465550184249878, 'val_mae': 0.8050989508628845}
{'epoch': 4, 'train_loss': 0.8794251997764707, 'train_nse': 0.12057480022352929, 'val_loss': 0.6280622183785645, 'val_nse': 0.37193778162143554, 'val_rmse': 3.4538605213165283, 'val_mae': 0.8085225224494934}
{'epoch': 5, 'train_loss': 0.8764508595814443, 'train_nse': 0.12354914041855569, 'val_loss': 0.6271075648332916, 'val_nse': 0.37289243516670845, 'val_rmse': 3.4512345790863

/Users/xshan/miniforge3/envs/cs7643/lib/python3.8/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


{'epoch': 1, 'train_loss': 0.8791755695658268, 'train_nse': 0.12082443043417324, 'val_loss': 0.5574493308467999, 'val_nse': 0.44255066915320007, 'val_rmse': 2.8730432987213135, 'val_mae': 0.8632111549377441}
{'epoch': 2, 'train_loss': 0.8280342507498827, 'train_nse': 0.17196574925011732, 'val_loss': 0.49681697832694804, 'val_nse': 0.503183021673052, 'val_rmse': 2.7122998237609863, 'val_mae': 0.8302576541900635}
{'epoch': 3, 'train_loss': 0.8173912443618684, 'train_nse': 0.18260875563813161, 'val_loss': 0.4830303446824844, 'val_nse': 0.5169696553175156, 'val_rmse': 2.6744019985198975, 'val_mae': 0.9044306874275208}
{'epoch': 4, 'train_loss': 0.8122457236214138, 'train_nse': 0.18775427637858622, 'val_loss': 0.4810281792400757, 'val_nse': 0.5189718207599243, 'val_rmse': 2.668853521347046, 'val_mae': 0.7904568314552307}
{'epoch': 5, 'train_loss': 0.8081804313717679, 'train_nse': 0.19181956862823213, 'val_loss': 0.45909285676041023, 'val_nse': 0.5409071432395898, 'val_rmse': 2.6072924137115

/Users/xshan/miniforge3/envs/cs7643/lib/python3.8/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


{'epoch': 1, 'train_loss': 0.9211783383388044, 'train_nse': 0.07882166166119564, 'val_loss': 0.7000781691974901, 'val_nse': 0.2999218308025099, 'val_rmse': 3.612414836883545, 'val_mae': 0.8268986940383911}
{'epoch': 2, 'train_loss': 0.8945019000308225, 'train_nse': 0.10549809996917747, 'val_loss': 0.6733407598147335, 'val_nse': 0.3266592401852665, 'val_rmse': 3.5427606105804443, 'val_mae': 0.8360594511032104}
{'epoch': 3, 'train_loss': 0.8879012919607019, 'train_nse': 0.11209870803929811, 'val_loss': 0.6625623179725898, 'val_nse': 0.3374376820274102, 'val_rmse': 3.5142910480499268, 'val_mae': 0.8198792934417725}
{'epoch': 4, 'train_loss': 0.8847675212047067, 'train_nse': 0.1152324787952933, 'val_loss': 0.6774118953514386, 'val_nse': 0.32258810464856136, 'val_rmse': 3.553454637527466, 'val_mae': 0.764268159866333}
{'epoch': 5, 'train_loss': 0.88214455278602, 'train_nse': 0.11785544721398, 'val_loss': 0.6654506889125926, 'val_nse': 0.3345493110874074, 'val_rmse': 3.5219428539276123, 'val

## Select Best Hyperparameters

In [ ]:
best_hparams = search_results_df.iloc[0].to_dict()
final_epochs = max(1, min(CONFIG['FINAL_MAX_EPOCHS'], int(round(best_hparams['cv_mean_best_epoch']))))
print('Best random-search configuration:')
print(best_hparams)
print('Final training epochs chosen from CV:', final_epochs)

## Final Training On All Pre-2008 Data

The final model trains on all pre-2008 windows from all selected basins and is evaluated once on the independent holdout period 2008-2014.

In [ ]:
final_train_map = make_target_map_for_fold(protocol, fold_id=0, split_name='pre_holdout_all')
final_holdout_map = make_target_map_for_fold(protocol, fold_id=0, split_name='holdout')
final_feature_mean, final_feature_std = feature_stats_from_target_map(
    basin_store,
    final_train_map,
    CONFIG['LOOKBACK_DAYS'],
    CONFIG['PREDICTION_HORIZON_DAYS'],
)

final_train_dataset = WindowedTargetDataset(
    basin_store=basin_store,
    target_map=final_train_map,
    lookback_days=CONFIG['LOOKBACK_DAYS'],
    horizon_days=CONFIG['PREDICTION_HORIZON_DAYS'],
    feature_mean=final_feature_mean,
    feature_std=final_feature_std,
)
final_holdout_dataset = WindowedTargetDataset(
    basin_store=basin_store,
    target_map=final_holdout_map,
    lookback_days=CONFIG['LOOKBACK_DAYS'],
    horizon_days=CONFIG['PREDICTION_HORIZON_DAYS'],
    feature_mean=final_feature_mean,
    feature_std=final_feature_std,
)

final_batch_size = int(best_hparams['batch_size'])
final_train_loader = make_dataloader(final_train_dataset, batch_size=final_batch_size, shuffle=True)
final_holdout_loader = make_dataloader(final_holdout_dataset, batch_size=final_batch_size, shuffle=False)

final_model = VanillaTransformerRegressor(
    num_features=len(feature_columns),
    d_model=int(best_hparams['d_model']),
    nhead=int(best_hparams['nhead']),
    num_layers=int(best_hparams['num_layers']),
    dim_feedforward=int(best_hparams['dim_feedforward']),
    dropout=float(best_hparams['dropout']),
).to(CONFIG['DEVICE'])

final_model, final_history_df = fit_model_fixed_epochs(
    model=final_model,
    train_loader=final_train_loader,
    learning_rate=float(best_hparams['learning_rate']),
    weight_decay=float(best_hparams['weight_decay']),
    epochs=final_epochs,
    device=CONFIG['DEVICE'],
)

holdout_metrics = run_epoch(final_model, final_holdout_loader, None, CONFIG['DEVICE'])
print('Independent holdout metrics:', holdout_metrics)

## Save Artifacts

In [ ]:
artifacts_dir = Path(CONFIG['OUTPUT_DIR'])
artifacts_dir.mkdir(parents=True, exist_ok=True)

config_path = artifacts_dir / 'vanilla_transformer_config.json'
search_path = artifacts_dir / 'vanilla_transformer_random_search_results.csv'
folds_path = artifacts_dir / 'vanilla_transformer_cv_fold_results.csv'
history_path = artifacts_dir / 'vanilla_transformer_final_history.csv'
metrics_path = artifacts_dir / 'vanilla_transformer_holdout_metrics.json'
checkpoint_path = artifacts_dir / 'vanilla_transformer_best.pt'

serializable_config = dict(CONFIG)
serializable_config['FEATURE_COLUMNS'] = feature_columns
serializable_config['TARGET_COLUMN'] = target_column
serializable_config['FINAL_EPOCHS_FROM_CV'] = final_epochs
serializable_config['PROTOCOL'] = {
    'holdout_start': protocol['holdout_start'],
    'holdout_end': protocol['holdout_end'],
    'cv_folds': protocol['cv_folds'],
}
config_path.write_text(json.dumps(serializable_config, indent=2), encoding='utf-8')
search_results_df.to_csv(search_path, index=False)
cv_fold_results_df.to_csv(folds_path, index=False)
final_history_df.to_csv(history_path, index=False)
metrics_path.write_text(json.dumps(holdout_metrics, indent=2), encoding='utf-8')

if CONFIG['SAVE_CHECKPOINTS']:
    torch.save(
        {
            'model_state_dict': final_model.state_dict(),
            'feature_columns': feature_columns,
            'target_column': target_column,
            'feature_mean': final_feature_mean,
            'feature_std': final_feature_std,
            'best_hparams': best_hparams,
            'config': serializable_config,
            'holdout_metrics': holdout_metrics,
        },
        checkpoint_path,
    )

print('Saved config:      ', config_path)
print('Saved search:      ', search_path)
print('Saved fold results:', folds_path)
print('Saved history:     ', history_path)
print('Saved holdout:     ', metrics_path)
if CONFIG['SAVE_CHECKPOINTS']:
    print('Saved checkpoint:  ', checkpoint_path)

## Plot Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
search_results_df.plot(x='trial', y='cv_mean_val_nse', marker='o', ax=axes[0])
axes[0].set_title('Random Search CV Mean NSE')
axes[0].set_ylabel('CV mean NSE')
axes[0].grid(True)

final_history_df.plot(x='epoch', y='train_nse', marker='o', ax=axes[1])
axes[1].set_title('Final Training NSE History')
axes[1].set_ylabel('Train NSE')
axes[1].grid(True)
plt.tight_layout()
plt.show()